# Rule-table pipeline starter, age ratings & broadcast restrictions (linear TV)Two things live in this notebook:1. **Cell 1-2**, convert `RULES_TABLE_v1.xlsx` into a real multi-tab **Google Sheet**   inside the Drive source folder. Run this once.2. **Cell 3-5**, the skeleton of the **ingestion pipeline**: hash each source document,   re-extract only what changed, version superseded rows, and run the integrity checks.Folder: `age_ratings_linear_tv_sources`Drive folder id: `1evp-ZPCqaycIU4ZGg9ERhs7O2SxwaNEW`

In [ ]:
from google.colab import authauth.authenticate_user()from googleapiclient.discovery import buildfrom googleapiclient.http import MediaFileUploaddrive = build('drive', 'v3')FOLDER_ID = '1evp-ZPCqaycIU4ZGg9ERhs7O2SxwaNEW'print('authenticated')

## 1. Convert the workbook to a multi-tab Google SheetUpload `RULES_TABLE_v1.xlsx` to the Colab session first (files pane, or `files.upload()`).Setting the target mimeType to `application/vnd.google-apps.spreadsheet` makes Drive convertit on the way in, so **all six tabs, the colour coding, the filters and the dropdowns survive**.

In [ ]:
XLSX = 'RULES_TABLE_v1.xlsx'          # path in the Colab sessioncreated = drive.files().create(    body={        'name': 'RULES_TABLE_v1',        'parents': [FOLDER_ID],        'mimeType': 'application/vnd.google-apps.spreadsheet',   # <- triggers conversion    },    media_body=MediaFileUpload(        XLSX,        mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet',        resumable=True),    fields='id,name,webViewLink',    supportsAllDrives=True,).execute()print(created['name'], '->', created['webViewLink'])SHEET_ID = created['id']

## 2. Read it back with gspreadFrom here on the sheet is the ground truth. Filter `status == 'approved'` before usingany row to score the prompt, `draft` and `unverified` rows have open conflicts(see the `Open_Questions` tab).

In [ ]:
import gspreadfrom google.auth import defaultimport pandas as pdcreds, _ = default()gc = gspread.authorize(creds)ws = gc.open_by_key(SHEET_ID).worksheet('Rules')rules = pd.DataFrame(ws.get_all_records())print(rules.shape)print(rules.groupby(['country', 'status']).size())print()print('usable as ground truth:', (rules.status == 'approved').sum())print('needs human review   :', (rules.human_review_required == 'yes').sum())approved = rules[rules.status == 'approved']

## 3. Source registry, hash-based change detectionThe point of the pipeline: a document is re-extracted **only when its bytes change**.`doc_hash` is what tells you that. Keep this table next to the rules.Put the official PDFs/HTML of each source in the Drive folder first, see `C-04` in`99_EXTRACTION_NOTES_AND_LIMITS.md`. Without local canonical copies the integritycheck in cell 5 cannot run.

In [ ]:
import hashlib, io, datetimefrom googleapiclient.http import MediaIoBaseDownloaddef drive_bytes(file_id):    buf = io.BytesIO()    dl = MediaIoBaseDownload(buf, drive.files().get_media(fileId=file_id))    done = False    while not done:        _, done = dl.next_chunk()    return buf.getvalue()def list_folder(folder_id):    out, token = [], None    while True:        r = drive.files().list(            q=f"'{folder_id}' in parents and trashed=false",            fields='nextPageToken, files(id,name,mimeType,size,modifiedTime)',            pageToken=token).execute()        out += r.get('files', [])        token = r.get('nextPageToken')        if not token:            break    return outregistry = []for f in list_folder(FOLDER_ID):    if f['mimeType'].startswith('application/vnd.google-apps'):        continue                      # native Google files have no stable bytes    raw = drive_bytes(f['id'])    registry.append({        'file_id':   f['id'],        'name':      f['name'],        'bytes':     len(raw),        'doc_hash':  hashlib.sha256(raw).hexdigest()[:16],        'seen_at':   datetime.date.today().isoformat(),    })registry = pd.DataFrame(registry)registry

## 4. Versioning a replaced documentWhen a source document changes, **never overwrite rows.** Close the old ones and append new ones,so you can still say which rule version produced a past decision.This is the behaviour the `valid_from` / `valid_to` / `status` columns exist for.

In [ ]:
def supersede(rules_df, source_doc, as_of=None):    """Close every live row from one source document. Returns the modified frame."""    as_of = as_of or datetime.date.today().isoformat()    hit = (rules_df.source_doc == source_doc) & (rules_df.valid_to == 'open')    rules_df.loc[hit, 'valid_to'] = as_of    rules_df.loc[hit, 'status']   = 'superseded'    print(f'superseded {hit.sum()} rows from {source_doc} as of {as_of}')    return rules_df# example (not run):# rules = supersede(rules, 'FR-R2')# rules = pd.concat([rules, newly_extracted_rows], ignore_index=True)

## 5. Integrity checks, run these before trusting the tableThese are the controls that stop a silent extraction error from becoming ground truth.Check 4 is the important one and it needs the official source files in the folder (see `C-04`).

In [ ]:
import reproblems = []# 1 - unique, well-formed rule idsdupes = rules.rule_id[rules.rule_id.duplicated()].tolist()if dupes:    problems.append(f'duplicate rule_id: {dupes}')bad_id = [r for r in rules.rule_id if not re.fullmatch(r'(UK|FR|EU)-[A-Z]{2,4}-\d{3}', r)]if bad_id:    problems.append(f'malformed rule_id: {bad_id}')# 2 - every row carries a source and a URLfor col in ('source_doc', 'clause_ref', 'source_url'):    missing = rules.loc[rules[col].astype(str).str.strip() == '', 'rule_id'].tolist()    if missing:        problems.append(f'{col} empty for: {missing}')# 3 - no row is simultaneously approved and flaggedcontradiction = rules[(rules.status == 'approved') &                      (rules.ambiguity_flag.isin(['translation_uncertain', 'unverified',                                                  'conflicting_sources']))]if len(contradiction):    problems.append('approved but flagged: ' + ', '.join(contradiction.rule_id))# 4 - machine time rules must parseTOKEN = re.compile(    r'^(none|prohibited|any_time|n/a|exempt=.+|see=.+|judgment:.+|framework:.+|gate=.+|'    r'not_before=\d{2}:\d{2}|window=\d{2}:\d{2}-\d{2}:\d{2}|'    r'restricted_window=\d{2}:\d{2}-\d{2}:\d{2}|soft_prefer_not_before=\d{2}:\d{2}|'    r'not_within=.+|not_near=.+|prohibited_in=.+|requires=.+|unless=.+|days=.+|'    r'except_days=.+|annual_quota=.+|sub_quota=.+|standard=.+|frequency_cap=.+|'    r'override=.+)$')for _, r in rules.iterrows():    for part in str(r.time_rule_machine).split(';'):        part = part.strip()        if part and not TOKEN.match(part):            problems.append(f'{r.rule_id}: unparseable time token {part!r}')# 5 - THE KEY CHECK (needs official source files in the folder)#     every verbatim_quote must actually occur in its stored source document.#     Until the PDFs are deposited this cannot run - that is conflict C-04.def check_quotes(rules_df, local_text_by_source):    fails = []    for _, r in rules_df.iterrows():        q = str(r.verbatim_quote).strip()        if not q or r.source_doc not in local_text_by_source:            continue        if q.replace('...', '') not in local_text_by_source[r.source_doc]:            fails.append((r.rule_id, q[:60]))    return failsprint(f'{len(problems)} problem(s)')for p in problems:    print(' -', p)

## 6. What is deliberately *not* automatedFrom `99_EXTRACTION_NOTES_AND_LIMITS.md` and the `Open_Questions` tab, these need a person:| Needs a human | Why ||---|---|| Confirming the French cat. III hours against the Legifrance original (**C-02**) | The Arcom English page is a machine translation and is garbled at art. 3. Highest-value bright lines in the table. || Confirming recommandation 2004-7 is still in force (**C-03**) | Sole source of the French -18 window; dates from 2004 with elapsed deadlines. || French **channel conventions** (**C-06**) | Per-licensee, not consolidated anywhere, can be stricter than the general rule. Cannot be scraped generically. || Calibrating `judgment_based` rows | Only Ofcom enforcement decisions give applied thresholds. Needs editorial judgement to turn cases into anchors. || Independently re-deriving the bright-line rows (**C-08**) | Guards against correlated error between the table and the prompt - the table must not be graded by its own author. |